In [1]:
import json
import os
from typing import Optional

import pandas as pd
import numpy as np
import torch

from logging_setup import create_logger
from phi_3_5_constants import hidden_state_size, seed, dsets_folder, token_lengths_path, train_split_records_path, \
    validation_split_records_path, directions_results_folder, finalized_activations_dir, dsets_index_path, \
    four_way_topics_index_path
from direction_learning import DirVectors, learn_directions_for_dset, VariantCombosInTopic

In [2]:
np_rng = np.random.default_rng(seed)
torch_rng = torch.Generator().manual_seed(seed)
torch.manual_seed(seed)

In [3]:
logger = create_logger(__name__)

In [5]:
directions_results_folder.mkdir(exist_ok=True)

In [6]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")

In [7]:
num_dsets = dsets_index_df.shape[0]

In [8]:
with token_lengths_path.open("r") as f:
    record_lengths_in_tokens = json.load(f)
with train_split_records_path.open("r") as f:
    train_split_record_idxs = json.load(f)
with validation_split_records_path.open("r") as f:
    validation_split_record_idxs = json.load(f)

In [9]:
all_dsets_activations: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]
all_dsets_labels: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]

all_dsets_train_split_polarity_labels: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]

In [ ]:
#storing dataset index numbers for later experiments that combine datasets

# top-level key is the name of a topic that has pos/neg/conj/disj variants, second level key is one of those variant names
dset_idxs_for_4way_topics: dict[str, dict[str, int]] = {}

idxs_for_other_dsets = dict[str, int] = {}

In [10]:
train_data_mean_activs_and_dirs_by_dset: list[Optional[DirVectors]] = [None for _ in range(num_dsets)]  

In [ ]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    categ_nm = dset_dtls["Categ_Folder"]
    dset_file_nm = dset_dtls["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    if dset_dtls['is_other']:
        idxs_for_other_dsets[dset_nm] = dset_idx
    else:
        if categ_nm not in dset_idxs_for_4way_topics.keys():
            dset_idxs_for_4way_topics[categ_nm] = {}
        if dset_dtls['is_negated']:
            dset_idxs_for_4way_topics[categ_nm]["neg"] = dset_idx
        elif dset_dtls['is_conj']:
            dset_idxs_for_4way_topics[categ_nm]["conj"] = dset_idx
        elif dset_dtls['is_disj']:
            dset_idxs_for_4way_topics[categ_nm]["disj"] = dset_idx
        else:
            dset_idxs_for_4way_topics[categ_nm]["affirm"] = dset_idx
    
    dataset = pd.read_csv(dsets_folder / categ_nm / dset_file_nm)
    dset_size = dataset.shape[0]
    dset_labels = torch.from_numpy(dataset['label'].to_numpy().astype(np.float32)[:, np.newaxis])
    all_dsets_labels[dset_idx] = dset_labels
    
    activs_path = finalized_activations_dir / categ_nm / (dset_nm + ".pt")
    relevant_activations = torch.load(activs_path, weights_only=True)
    assert relevant_activations.shape == (2, dset_size, hidden_state_size)
    all_dsets_activations[dset_idx] = relevant_activations
    
    train_split_activs = relevant_activations[:, train_split_record_idxs[str(dset_idx)], :]
    train_split_truth_labels = dset_labels[train_split_record_idxs[str(dset_idx)], :]
    
    train_split_polarity_labels = torch.ones(train_split_truth_labels.shape)
    if dset_dtls["is_negated"]:
        train_split_polarity_labels *= -1
    all_dsets_train_split_polarity_labels[dset_idx] = train_split_polarity_labels
    
    # val_split_activs = relevant_activations[:, validation_split_record_idxs[str(dset_idx)], :]
    # val_split_truth_labels = dset_labels[validation_split_record_idxs[str(dset_idx)], :] 
    
    dir_vects_for_dset = learn_directions_for_dset(
        directions_results_folder / categ_nm, dset_nm, train_split_activs, train_split_truth_labels,
        train_split_polarity_labels, np_rng)
    train_data_mean_activs_and_dirs_by_dset[dset_idx] = dir_vects_for_dset

assert all([len(idxs_of_4way_topic) == 4 for idxs_of_4way_topic in dset_idxs_for_4way_topics.values()])
with four_way_topics_index_path.open("w") as f:
    json.dump(dset_idxs_for_4way_topics, f)

In [ ]:
# top-level key is a category name (specifically a category/topic name that has the 4 variants affirmative/negated/conjunctive/disjunctive
train_data_multi_topic_variants_mean_activs_and_dirs: dict[str, VariantCombosInTopic] = {}

In [ ]:
for topic_nm, variants_dset_idxs in dset_idxs_for_4way_topics.items():
    affirm_idx = variants_dset_idxs["affirm"]
    affirm_train_activs = all_dsets_activations[affirm_idx][:, train_split_record_idxs[str(affirm_idx)], :]
    affirm_train_truth_labels = all_dsets_labels[affirm_idx][train_split_record_idxs[str(affirm_idx)], :]
    affirm_train_polarity_labels = all_dsets_train_split_polarity_labels[affirm_idx] 
    
    neg_idx = variants_dset_idxs["neg"]
    neg_train_activs = all_dsets_activations[neg_idx][:, train_split_record_idxs[str(neg_idx)], :]
    neg_train_truth_labels = all_dsets_labels[neg_idx][train_split_record_idxs[str(neg_idx)], :]
    neg_train_polarity_labels = all_dsets_train_split_polarity_labels[neg_idx]
    
    conj_idx = variants_dset_idxs["conj"]
    conj_train_activs = all_dsets_activations[conj_idx][:, train_split_record_idxs[str(conj_idx)], :]
    conj_train_truth_labels = all_dsets_labels[conj_idx][train_split_record_idxs[str(conj_idx)], :]
    conj_train_polarity_labels = all_dsets_train_split_polarity_labels[conj_idx]
    
    disj_idx = variants_dset_idxs["disj"]
    disj_train_activs = all_dsets_activations[disj_idx][:, train_split_record_idxs[str(disj_idx)], :]
    disj_train_truth_labels = all_dsets_labels[disj_idx][train_split_record_idxs[str(disj_idx)], :]
    disj_train_polarity_labels = all_dsets_train_split_polarity_labels[disj_idx]
    
    #affirm_neg
    affirm_neg_dirs = learn_directions_for_dset(
        directions_results_folder / topic_nm, "affirm_neg", torch.concat((affirm_train_activs, neg_train_activs), dim=1),
        torch.concat((affirm_train_truth_labels, neg_train_truth_labels), dim=0), 
        torch.concat((affirm_train_polarity_labels, neg_train_polarity_labels), dim=0), np_rng)
    
    #affirm_disj
    affirm_disj_dirs = learn_directions_for_dset(
        directions_results_folder / topic_nm, "affirm_disj", torch.concat((affirm_train_activs, disj_train_activs), dim=1),
        torch.concat((affirm_train_truth_labels, disj_train_truth_labels), dim=0),
        torch.concat((affirm_train_polarity_labels, disj_train_polarity_labels), dim=0), np_rng)
    
    # neg_conj
    neg_conj_dirs = learn_directions_for_dset(
        directions_results_folder / topic_nm, "neg_conj", torch.concat((neg_train_activs, conj_train_activs), dim=1),
        torch.concat((neg_train_truth_labels, conj_train_truth_labels), dim=0),
        torch.concat((neg_train_polarity_labels, conj_train_polarity_labels), dim=0), np_rng)
    
    # affirm_neg_conj_disj
    affirm_neg_conj_disj_dirs = learn_directions_for_dset(
        directions_results_folder / topic_nm, "affirm_neg_conj_disj", 
        torch.concat((affirm_train_activs, neg_train_activs, conj_train_activs, disj_train_activs), dim=1),
        torch.concat((affirm_train_truth_labels, neg_train_truth_labels, conj_train_truth_labels, disj_train_truth_labels
                      ), dim=0),
        torch.concat((affirm_train_polarity_labels, neg_train_polarity_labels, conj_train_polarity_labels,
                      disj_train_polarity_labels), dim=0), np_rng)
    
    train_data_multi_topic_variants_mean_activs_and_dirs[topic_nm] = VariantCombosInTopic(
        affirm_neg_dirs, affirm_disj_dirs, neg_conj_dirs, affirm_neg_conj_disj_dirs)

In [ ]:
unambig_lie_idx = idxs_for_other_dsets["unambiguous_lie"]
unambig_lie_train_activs = all_dsets_activations[unambig_lie_idx][:, train_split_record_idxs[str(unambig_lie_idx)], :]
unambig_lie_train_truth_labels = all_dsets_labels[unambig_lie_idx][train_split_record_idxs[str(unambig_lie_idx)], :]
unambig_lie_train_polarity_labels = all_dsets_train_split_polarity_labels[unambig_lie_idx]

unambig_truth_idx = idxs_for_other_dsets["unambiguous_truthful_reply"]
unambig_truth_train_activs = all_dsets_activations[unambig_truth_idx][:, train_split_record_idxs[str(unambig_truth_idx)], :]
unambig_truth_train_truth_labels = all_dsets_labels[unambig_truth_idx][train_split_record_idxs[str(unambig_truth_idx)], :]
unambig_truth_train_polarity_labels = all_dsets_train_split_polarity_labels[unambig_truth_idx]

ambig_truth_idx = idxs_for_other_dsets["ambiguous_truthful_reply"]
ambig_truth_train_activs = all_dsets_activations[ambig_truth_idx][:, train_split_record_idxs[str(ambig_truth_idx)], :]
ambig_truth_train_truth_labels = all_dsets_labels[ambig_truth_idx][train_split_record_idxs[str(ambig_truth_idx)], :]
ambig_truth_train_polarity_labels = all_dsets_train_split_polarity_labels[ambig_truth_idx]

selected_real_world_train_activs = torch.concat(
    (unambig_lie_train_activs, unambig_truth_train_activs, ambig_truth_train_activs), dim=1)
selected_real_world_train_truth_labels = torch.concat(
    (unambig_lie_train_truth_labels, unambig_truth_train_truth_labels, ambig_truth_train_truth_labels), dim=0)
selected_real_world_train_polarity_labels = torch.concat(
    (unambig_lie_train_polarity_labels, unambig_truth_train_polarity_labels, ambig_truth_train_polarity_labels), dim=0)
real_world_multi_dset_dirs = learn_directions_for_dset(
    directions_results_folder / "real_world_scenarios", "unambig_lie_unambig_truth_ambig_truth",
    selected_real_world_train_activs,selected_real_world_train_truth_labels, selected_real_world_train_polarity_labels,
    np_rng)

In [ ]:
#  I think it makes sense to do affirm+neg+conj, so we can see if a very broad training corpus allows good generalization to disj; it also differentiates this more from the final scenario (affirm/neg for that kind of topic plus some of the 'other' datasets)
multi_topics_affirm_neg_conj_train_activs = torch.zeros((2, 1, hidden_state_size))
multi_topics_affirm_neg_conj_train_truth_labels = torch.zeros((1, 1))
multi_topics_affirm_neg_conj_train_polarity_labels = torch.zeros((1, 1))

#these will be part of a final experiment that also includes some of the 'other' datasets
various_categories_train_activs = torch.zeros((2, 1, hidden_state_size))
various_categories_train_truth_labels = torch.zeros((1, 1))
various_categories_train_polarity_labels = torch.zeros((1, 1))

for topic_nm, variants_dset_idxs in dset_idxs_for_4way_topics.items():
    if topic_nm not in ("animal_class", "element_symbols", "facts", "inventors"):
        continue
    logger.info(f"concatenating data for the topic {topic_nm} to prepare for multi-topic/category experiments")
    affirm_idx = variants_dset_idxs["affirm"]
    affirm_train_activs = all_dsets_activations[affirm_idx][:, train_split_record_idxs[str(affirm_idx)], :]
    affirm_train_truth_labels = all_dsets_labels[affirm_idx][train_split_record_idxs[str(affirm_idx)], :]
    affirm_train_polarity_labels = all_dsets_train_split_polarity_labels[affirm_idx]
    
    neg_idx = variants_dset_idxs["neg"]
    neg_train_activs = all_dsets_activations[neg_idx][:, train_split_record_idxs[str(neg_idx)], :]
    neg_train_truth_labels = all_dsets_labels[neg_idx][train_split_record_idxs[str(neg_idx)], :]
    neg_train_polarity_labels = all_dsets_train_split_polarity_labels[neg_idx]
    
    conj_idx = variants_dset_idxs["conj"]
    conj_train_activs = all_dsets_activations[conj_idx][:, train_split_record_idxs[str(conj_idx)], :]
    conj_train_truth_labels = all_dsets_labels[conj_idx][train_split_record_idxs[str(conj_idx)], :]
    conj_train_polarity_labels = all_dsets_train_split_polarity_labels[conj_idx]

    multi_topics_affirm_neg_conj_train_activs = torch.concat((
        multi_topics_affirm_neg_conj_train_activs, affirm_train_activs, neg_train_activs, conj_train_activs), dim=1)
    multi_topics_affirm_neg_conj_train_truth_labels = torch.concat((
        multi_topics_affirm_neg_conj_train_truth_labels, affirm_train_truth_labels, neg_train_truth_labels,
        conj_train_truth_labels), dim=0)
    multi_topics_affirm_neg_conj_train_polarity_labels = torch.concat((
        multi_topics_affirm_neg_conj_train_polarity_labels, affirm_train_polarity_labels, neg_train_polarity_labels,
        conj_train_polarity_labels), dim=0)
    
    various_categories_train_activs = torch.concat((
        various_categories_train_activs, affirm_train_activs, neg_train_activs), dim=1)
    various_categories_train_truth_labels = torch.concat((
        various_categories_train_truth_labels, affirm_train_truth_labels, neg_train_truth_labels), dim=0)
    various_categories_train_polarity_labels = torch.concat((
        various_categories_train_polarity_labels, affirm_train_polarity_labels, neg_train_polarity_labels), dim=0)

multi_topics_affirm_neg_conj_train_activs = multi_topics_affirm_neg_conj_train_activs[:, 1:, :]
multi_topics_affirm_neg_conj_train_truth_labels = multi_topics_affirm_neg_conj_train_truth_labels[1:, :]
multi_topics_affirm_neg_conj_train_polarity_labels = multi_topics_affirm_neg_conj_train_polarity_labels[1:, :]

various_categories_train_activs = various_categories_train_activs[:, 1:, :]
various_categories_train_truth_labels = various_categories_train_truth_labels[1:, :]
various_categories_train_polarity_labels = various_categories_train_polarity_labels[1:, :]

In [ ]:
multi_topics_affirm_neg_conj_dirs = learn_directions_for_dset(
    directions_results_folder, "multi_topics_affirm_neg_conj", multi_topics_affirm_neg_conj_train_activs,
    multi_topics_affirm_neg_conj_train_truth_labels, multi_topics_affirm_neg_conj_train_polarity_labels, np_rng)

In [ ]:
#  affirmative and negated statements from "animal class", "element symbols", "facts", and "inventors"
#  plus "unambiguous lie", "unambiguous truthful reply", and "ambiguous truthful reply" from 'real world scenarios'
#  plus "smaller than" from 'relative comparison'
#  plus "common claim" from 'true false'

smaller_than_idx = idxs_for_other_dsets["smaller_than"]
smaller_than_train_activs = all_dsets_activations[smaller_than_idx][:, train_split_record_idxs[str(smaller_than_idx)], :]
smaller_than_train_truth_labels = all_dsets_labels[smaller_than_idx][train_split_record_idxs[str(smaller_than_idx)], :]
smaller_than_train_polarity_labels = all_dsets_train_split_polarity_labels[smaller_than_idx]

common_claim_idx = idxs_for_other_dsets["common_claim_true_false"]
common_claim_train_activs = all_dsets_activations[common_claim_idx][:, train_split_record_idxs[str(common_claim_idx)], :]
common_claim_train_truth_labels = all_dsets_labels[common_claim_idx][train_split_record_idxs[str(common_claim_idx)], :]
common_claim_train_polarity_labels = all_dsets_train_split_polarity_labels[common_claim_idx]

various_categories_train_activs = torch.concat((
    various_categories_train_activs, selected_real_world_train_activs, smaller_than_train_activs, 
    common_claim_train_activs), dim=1)
various_categories_train_truth_labels = torch.concat((
    various_categories_train_truth_labels, selected_real_world_train_truth_labels, smaller_than_train_truth_labels,
    common_claim_train_truth_labels), dim=0)
various_categories_train_polarity_labels = torch.concat((
    various_categories_train_polarity_labels, selected_real_world_train_polarity_labels, smaller_than_train_polarity_labels,
    common_claim_train_polarity_labels), dim=0)

various_categories_dirs = learn_directions_for_dset(
    directions_results_folder, "various_categories", various_categories_train_activs,
    various_categories_train_truth_labels, various_categories_train_polarity_labels, np_rng)

In [19]:
# for dset_idx, dset_dtls in dsets_index_df.iterrows():
#     if dset_idx > 1:
#         break
#     categ_nm = dset_dtls["Categ_Folder"]
#     dset_file_nm = dset_dtls["Dataset_File"]
#     dset_nm = os.path.splitext(dset_file_nm)[0]
#     
#     


In [87]:
# all_dsets_activations[3].shape

torch.Size([2, 500, 3072])

In [10]:
#TODO train on positive + negated dataset pairs (within a category) to confirm whether polarity direction is doing any good 

np.int64(13)

In [68]:
# dummy_truth_probe_animals_pos_lyr_18 = PolarityAwareTruthProbe(torch.ones(hidden_state_size, 1), torch.ones(hidden_state_size,1), torch.ones(hidden_state_size,1))
#NOTE FOR LATER- loading model state dict worked if the probe's buffers had been initialized with all-ones tensors but didn't work in practice if they'd been initialized with all-zeros tensors (everything would look right when examining the loaded probe but any predictions it made would be all nan)
# That is, the buffers of the probe object which was created solely so that its load_state_dict() method could be called

In [70]:
# dummy_truth_probe_animals_pos_lyr_18.load_state_dict(torch.load(results_folder / "animal_class" / "animal_class_lyr18.pth"))

C:\Users\ssili\AppData\Local\Temp\ipykernel_55864\4070788335.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dummy_truth_probe_animals_pos_lyr_18.load_state_dict(torch.l

<All keys matched successfully>

In [71]:
# dummy_truth_probe_animals_pos_lyr_18(all_dsets_activations[0][0, 0:2, :])

tensor([[9.9996e-01],
        [2.6754e-05]], grad_fn=<SigmoidBackward0>)

In [88]:
# activations_ = all_dsets_activations[2][0]
# print(f"activs type={type(activations_)}; shape={activations_.shape}, dtype={activations_.dtype}")
# print(activations_.mean(dim=0))
# preds = dummy_truth_probe_animals_pos_lyr_18(activations_)
# print(f"preds type={type(preds)}; shape={preds.shape}, dtype={preds.dtype}")
# #print(preds)
# labels = torch.from_numpy(all_dsets_labels[2][:, np.newaxis])
# print(f"labels type={type(labels)}; shape={labels.shape}")
# errs = preds - labels
# print(f"errs type={type(errs)}; shape={errs.shape}")
# (MSE := torch.mean(torch.square(errs)).item())

activs type=<class 'torch.Tensor'>; shape=torch.Size([500, 3072]), dtype=torch.float32
tensor([ 0.4619, -0.0388,  0.2940,  ...,  0.6617,  0.0588, -0.9982])
preds type=<class 'torch.Tensor'>; shape=torch.Size([500, 1]), dtype=torch.float32
labels type=<class 'torch.Tensor'>; shape=torch.Size([500, 1])
errs type=<class 'torch.Tensor'>; shape=torch.Size([500, 1])


0.34377041459083557